# Bài 03 · Mạng nhiều lớp & bài toán XOR: một lớp ẩn làm được điều một nơ-ron không làm được

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/03-mlp-va-xor.ipynb)

Notebook đi kèm [Bài 03 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/mlp-va-xor/) (reel và demo tương tác nằm ở trang bài học).
Bài 01 kết thúc ở một thất bại: không perceptron nào giải được XOR. Ở đây ta tự viết bằng numpy một mạng có
**lớp ẩn**, xem nó biến đổi bốn điểm XOR thế nào, cho nó tự học, rồi nhìn lớp ẩn của mạng MNIST thật.

**Bạn sẽ làm:**
1. Chứng minh không đường thẳng nào giải được XOR, và thấy quy tắc perceptron xoay vòng mãi trên nó.
2. Tự đặt trọng số cho một mạng 2 → 2 → 1 giải XOR, và nhìn bốn điểm trong **không gian ẩn**.
3. Cho mạng tự học trọng số với 100 khởi tạo khác nhau, và xem khi nào nó bị kẹt.
4. Thấy nhiều nơ-ron ẩn vẽ được vùng phức tạp hơn, và độ sâu nhân số vùng nhanh hơn độ rộng.
5. Huấn luyện mạng MNIST 784 → 128 → 64 → 10 của khóa học, rồi đo xem lớp ẩn của nó giúp perceptron Bài 01 đến đâu.

**Cần biết trước:** [Bài 01 · Perceptron](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/perceptron/) (quy tắc
học, $z = xW + b$) và [Bài 02 · Hàm kích hoạt](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-kich-hoat/)
(sigmoid, ReLU, vì sao cần $f$ phi tuyến). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong reel và demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import gzip
import hashlib
import itertools
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm hình dạng: nhãn 0 = tròn cam, nhãn 1 = vuông xanh
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # 0 → ½ → 1
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 109.184 · 0,5 · −0,13 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def vn_axes(fig):
    """Đổi nhãn vạch của mọi trục tuyến tính (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show()."""
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            if axis.get_scale() == "linear":
                axis.set_major_formatter(FuncFormatter(vn_tick))


print(vn(109184, 0), vn(0.5, 1), vn(-0.1263, 2), vn(-2.8e-17, 1), pct(0.9727))

## 2 · Dữ liệu: bốn điểm XOR (và MNIST cho mục 8)

XOR (“hoặc loại trừ”) cho 1 khi hai đầu vào **khác nhau**. Cả bộ dữ liệu chỉ có bốn điểm. Ta xếp chúng thành các
hàng của ma trận $P$, như mọi tập dữ liệu trong khóa học: mỗi hàng là một vectơ hàng $x = (x_1, x_2)$.

In [ ]:
P = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)  # hàng i: đầu vào (x₁, x₂) của điểm i
Y = np.array([0, 1, 1, 0])  # nhãn: 1 khi x₁ ≠ x₂
for i, (p, y) in enumerate(zip(P, Y)):
    print(f"điểm {i}: x = ({p[0]:.0f}, {p[1]:.0f})  →  y = {y}")


def show_points(ax, X=P, y=Y, s=170, legend=True):
    """Vẽ các điểm: nhãn 0 = tròn cam, nhãn 1 = vuông xanh (màu luôn đi kèm hình dạng)."""
    for (a, b), lab in zip(X, y):
        ax.scatter(a, b, s=s, marker="s" if lab else "o", color=BLUE if lab else ORANGE, ec="white", lw=2, zorder=3)
    if legend:
        ax.scatter([], [], s=80, marker="o", color=ORANGE, label="y = 0")
        ax.scatter([], [], s=80, marker="s", color=BLUE, label="y = 1")
        ax.legend(frameon=False, loc="upper right", fontsize=9)


fig, ax = plt.subplots(figsize=(3.6, 3.6))
show_points(ax)
for (a, b), lab in zip(P, Y):
    ax.annotate(f"({a:.0f}, {b:.0f})", (a, b), xytext=(0, -20), textcoords="offset points", ha="center", fontsize=9)
ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="x₁", ylabel="x₂", title="XOR: bốn điểm, hai nhãn")
ax.set_aspect("equal")
vn_axes(plt.gcf())
plt.show()

Hai điểm cam nằm trên một đường chéo, hai điểm xanh trên đường chéo kia.

Mục 8 dùng MNIST (70.000 ảnh chữ số viết tay 28×28), tải đúng như Bài 01 và kiểm tra mã MD5.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images / 255.0, labels.astype(np.int64)  # mỗi hàng là một ảnh: 784 số trong [0, 1]


fetch()
Xtr, ytr = load("train")
Xte, yte = load("t10k")
print("huấn luyện:", Xtr.shape, " kiểm tra:", Xte.shape)

## 3 · Nhắc lại Bài 01: một nơ-ron thua XOR

Ta chạy lại đúng thí nghiệm cuối Bài 01, cũng là thí nghiệm trong reel và demo: perceptron xuất phát từ
$w = (0{,}3;\ -0{,}2)$, $b = 0{,}1$, tốc độ học $\eta = 0{,}1$. Mỗi bước sửa điểm sai đầu tiên tính từ sau điểm vừa sửa.

Hàm `perceptron()` dưới đây dùng lại được cho mọi tập điểm nhỏ (ta sẽ cần nó ở mục 5). Hàm `z_of()` cộng theo đúng
thứ tự của reel và demo, $x_1 w_1 + x_2 w_2 + b$: vài lần sửa đưa $z$ về sát 0, và khi đó sai số làm tròn của số thực
quyết định dấu của $z$.

In [ ]:
def step(z):
    return (np.asarray(z) >= 0).astype(int)


def z_of(X, w, b):
    """z = x₁w₁ + x₂w₂ + … + b cho mọi hàng của X."""
    z = X[:, 0] * w[0]
    for i in range(1, X.shape[1]):
        z = z + X[:, i] * w[i]
    return z + b


def perceptron(X, y, w, b, eta=0.1, max_updates=16):
    """Quy tắc Rosenblatt (Bài 01). Trả về các trạng thái (w, b, số điểm đúng, con trỏ) sau mỗi lần sửa."""
    w, n, cursor = np.array(w, float), len(y), 0
    states = [(w.copy(), b, int((step(z_of(X, w, b)) == y).sum()), cursor)]
    for _ in range(max_updates):
        wrong = step(z_of(X, w, b)) != y
        if not wrong.any():  # đúng hết: dừng
            break
        j = next((cursor + q) % n for q in range(n) if wrong[(cursor + q) % n])
        d = y[j] - step(z_of(X[j : j + 1], w, b))[0]  # +1 hoặc −1
        w, b = w + eta * d * X[j], b + eta * d
        cursor = (j + 1) % n
        states.append((w.copy(), b, int((step(z_of(X, w, b)) == y).sum()), cursor))
    return states


run = perceptron(P, Y, [0.3, -0.2], 0.1)
history = [s[2] for s in run]
print("số điểm đúng sau mỗi lần sửa:", history)
assert history == [1, 1, 2, 2, 3, 1, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2], "khác với reel và demo: hãy kiểm tra lại"

Đúng dãy số của reel và demo: không bao giờ quá 3/4.

### Chạy lâu hơn thì sao?
Reel chỉ dừng sau 16 lần sửa. Ô dưới cho chạy 1.000 lần sửa và tìm xem trạng thái của thuật toán (gồm $w$, $b$ và điểm
sẽ xét tiếp) có bao giờ lặp lại không.

In [ ]:
def find_cycle(states):
    """(t₀, chu kỳ): lần sửa đầu tiên t₀ mà toàn bộ trạng thái (w, b, con trỏ) về sau lặp lại y hệt."""
    seen = {}
    for t, (w, b, _, cursor) in enumerate(states):
        key = (*w, b, cursor)
        if key in seen:
            return seen[key], t - seen[key]
        seen[key] = t
    return None


long_run = perceptron(P, Y, [0.3, -0.2], 0.1, max_updates=1000)
t0, period = find_cycle(long_run)
print(f"{vn(len(long_run) - 1, 0)} lần sửa; nhiều nhất {max(s[2] for s in long_run)}/4 điểm đúng")
print(f"từ lần sửa {t0}, trạng thái lặp lại y hệt sau mỗi {period} lần sửa:")
for t in range(t0, t0 + period):
    w_t, b_t, n_ok, _ = long_run[t]
    print(f"  sau lần sửa {t:>2}: w = ({vn(w_t[0], 1)}; {vn(w_t[1], 1)}), b = {vn(b_t, 1)}  →  đúng {n_ok}/4")
assert max(s[2] for s in long_run) < 4 and (t0, period) == (11, 4)

# Cùng thuật toán trong số học chính xác: nhân mọi thứ với 10 (w = (3; −2), b = 1, η = 1), chỉ còn số nguyên
exact = perceptron(P, Y, [3, -2], 1, eta=1, max_updates=1000)
print("số nguyên, 16 lần sửa đầu:", [s[2] for s in exact[:17]])
print(f"số nguyên: nhiều nhất {max(s[2] for s in exact)}/4 đúng; chu kỳ (t₀, độ dài) = {find_cycle(exact)}")
z_bad = z_of(P[3:], *long_run[3][:2])[0]
print("z của điểm (1, 1) sau lần sửa 3, tính bằng số thực:", f"{z_bad:.1e}".replace(".", ",").replace("-", "−"))
assert max(s[2] for s in exact) < 4

Không bao giờ đúng cả 4. Hơn thế, từ lần sửa thứ 11 thuật toán đi vào một **vòng lặp**: bốn trạng thái lặp lại y hệt
mãi mãi, $w$ không lớn dần mà cũng không về 0. Đây là định lý “xoay vòng” của Block & Levin (1970): khi dữ liệu không
tách được, trọng số của perceptron luôn bị chặn. Ở đây $x$ chỉ gồm 0 và 1, nên mỗi lần sửa cộng hoặc trừ đúng
$0{,}1$ vào vài trọng số; bị chặn thì chỉ có hữu hạn trạng thái, và thuật toán buộc phải quay lại một trạng thái cũ.

Ba dòng cuối của ô trên cho thấy một chi tiết nhỏ: trong số học chính xác (nhân mọi thứ lên 10 để chỉ còn số nguyên),
dãy số điểm đúng khác dãy của reel ngay sau lần sửa thứ 3. Sau lần sửa 3, điểm $(1, 1)$ có $z$ đúng ra bằng 0, nhưng số thực tính
ra một số âm rất nhỏ, nên dấu của $z$ đổi theo sai số làm tròn. Kết luận thì không đổi: cả hai đều xoay vòng mà không bao
giờ đúng 4/4.

### Vì sao không đường thẳng nào làm được
Một nơ-ron đoán 1 khi $z = x_1 w_1 + x_2 w_2 + b \ge 0$. Để đúng cả bốn điểm XOR, cần đồng thời:

$$\underbrace{b < 0}_{(0,0)\to 0}, \qquad \underbrace{w_2 + b \ge 0}_{(0,1)\to 1}, \qquad
\underbrace{w_1 + b \ge 0}_{(1,0)\to 1}, \qquad \underbrace{w_1 + w_2 + b < 0}_{(1,1)\to 0}.$$

Cộng hai bất đẳng thức giữa: $w_1 + w_2 + 2b \ge 0$, tức $w_1 + w_2 + b \ge -b > 0$, trái với bất đẳng thức cuối.
Không có $(w_1, w_2, b)$ nào thỏa cả bốn. Thử kiểm tra bằng 100.000 đường thẳng ngẫu nhiên:

In [ ]:
lines = np.random.default_rng(0).normal(size=(100_000, 3))  # mỗi hàng: (w₁, w₂, b) của một đường thẳng
n_correct = (step(P @ lines[:, :2].T + lines[:, 2]) == Y[:, None]).sum(axis=0)
print(f"{vn(len(lines), 0)} đường thẳng ngẫu nhiên:")
for k, count in enumerate(np.bincount(n_correct, minlength=5)):
    print(f"đúng {k}/4 điểm: {vn(count, 0):>6} đường thẳng")
assert n_correct.max() == 3 and n_correct.min() == 1

Không đường nào đúng 4/4, và cũng không đường nào sai cả 4 (sai hết nghĩa là giải được bài toán ngược lại, XNOR,
cũng không tách được). Đường “tốt nhất” chỉ đúng 3/4.

### Minsky & Papert (1969)
Cuốn *Perceptrons* của Marvin Minsky và Seymour Papert nghiên cứu chính xác điều này: một lớp học được, đặt trên các
đặc trưng cố định, tính được những gì? XOR là trường hợp hai đầu vào của hàm **chẵn lẻ** (parity: số đầu vào bằng 1 là
lẻ hay chẵn). Định lý của họ nói perceptron chỉ tính được parity của $n$ điểm ảnh khi có ít nhất một đặc trưng nhìn
thấy cả $n$ điểm. Cuốn sách không nói mạng nhiều lớp không giải được XOR: người ta đã biết từ McCulloch & Pitts (1943)
rằng mạng gồm các nơ-ron ngưỡng tính được mọi hàm logic. Điều còn thiếu là một cách **học** trọng số của các lớp ẩn,
và Minsky & Papert hoài nghi rằng có cách như vậy. Câu chuyện “một cuốn sách giết chết mạng nơ-ron” là một sự đơn giản
hóa, nhưng cuốn sách đúng là góp phần làm hướng nghiên cứu này chững lại, cho đến khi lan truyền ngược (Rumelhart,
Hinton & Williams, 1986; [Bài 06](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/)) phổ biến.

## 4 · Thêm một lớp ẩn: hai nơ-ron vẽ hai đường

Một đường không đủ, vậy dùng hai. Đặt hai nơ-ron cạnh nhau như cuối Bài 01, mỗi nơ-ron một cột của $W_1$:

$$h = f(x W_1 + b_1), \qquad \hat y = f(h W_2 + b_2)$$

Ở đây $x$ là vectơ hàng $1 \times 2$, $W_1$ cỡ $2 \times 2$, nên $h = (h_1, h_2)$ có hai số: đó là **lớp ẩn** (hidden
layer), vì không ai nhìn thấy hay cho nhãn nó. $W_2$ cỡ $2 \times 1$ nối $h$ với nơ-ron đầu ra. Ta tự chọn trọng số,
với $f$ là hàm bậc thang của Bài 01:

- $h_1 = f(x_1 + x_2 - 0{,}5)$: bằng 1 khi **ít nhất một** đầu vào bằng 1 (OR).
- $h_2 = f(x_1 + x_2 - 1{,}5)$: bằng 1 khi **cả hai** đầu vào bằng 1 (AND).
- $\hat y = f(h_1 - 2h_2 - 0{,}5)$: “OR nhưng không AND”, chính là XOR.

In [ ]:
W1 = np.array([[1.0, 1.0], [1.0, 1.0]])  # cột 1: nơ-ron h₁ (OR), cột 2: nơ-ron h₂ (AND)
b1 = np.array([-0.5, -1.5])
W2 = np.array([[1.0], [-2.0]])  # 2 × 1
b2 = np.array([-0.5])

H_hand = step(P @ W1 + b1)  # lớp ẩn của cả 4 điểm cùng lúc: Z = PW₁ + b₁, cỡ 4 × 2
y_hand = step(H_hand @ W2 + b2)[:, 0]
for p, h, yh, y in zip(P, H_hand, y_hand, Y):
    print(f"x = ({p[0]:.0f}, {p[1]:.0f})  →  h = ({h[0]}, {h[1]})  →  ŷ = {yh}   (nhãn {y})")
assert (y_hand == Y).all()
n_params = W1.size + b1.size + W2.size + b2.size
print(f"đúng {(y_hand == Y).sum()}/4 với {n_params} tham số; MAC mỗi điểm: {W1.size} + {W2.size} = {W1.size + W2.size}")

Đúng cả bốn điểm, với 9 tham số. Mỗi nơ-ron ẩn vẫn chỉ là một perceptron, tức một đường thẳng $z = 0$ trong mặt
phẳng $(x_1, x_2)$. Cả mạng kết hợp hai đường đó.

## 5 · Không gian ẩn: XOR trở nên tách được

### 🤔 Dự đoán trước
Lớp ẩn biến mỗi điểm $x$ thành một điểm mới $h = (h_1, h_2)$. Trong mặt phẳng $(h_1, h_2)$, bốn điểm XOR nằm ở đâu:
vẫn là bốn góc của hình vuông, hay khác đi? Và perceptron của Bài 01, xuất phát từ đúng
$w = (0{,}3;\ -0{,}2)$, $b = 0{,}1$, cần bao nhiêu lần sửa trên các điểm $h$ này?

In [ ]:
hidden_run = perceptron(H_hand.astype(float), Y, [0.3, -0.2], 0.1)
w_h, b_h, n_ok, _ = hidden_run[-1]
print("4 điểm trong không gian ẩn:", [tuple(int(v) for v in h) for h in H_hand])
print(
    f"perceptron trên h: {len(hidden_run) - 1} lần sửa → đúng {n_ok}/4  (w = ({vn(w_h[0], 1)}; {vn(w_h[1], 1)}), b = {vn(b_h, 1)})"
)
assert n_ok == 4 and len(hidden_run) - 1 == 2

Lớp ẩn **gộp** hai điểm $(0,1)$ và $(1,0)$ vào cùng một điểm $h = (1, 0)$. Bốn góc hình vuông chỉ còn ba điểm, và một
đường thẳng tách được chúng: chỉ cần 2 lần sửa, từ đúng điểm xuất phát đã thất bại ở mục 3.

Hình dưới đặt hai không gian cạnh nhau. Bên trái là không gian đầu vào $(x_1, x_2)$: nét gạch là đường $h_1$ (OR),
nét chấm là đường $h_2$ (AND), nền tô theo dự đoán của cả mạng (xanh: $\hat y = 1$, cam: $\hat y = 0$). Vùng đoán 1 là
dải nằm **giữa** hai đường, thứ một đường thẳng không vẽ được. Bên phải là không gian ẩn $(h_1, h_2)$ với đường
quyết định của nơ-ron đầu ra (xanh lá); số bên cạnh mỗi điểm cho biết bao nhiêu điểm XOR rơi vào đó.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-0.5, 1.5, 201), np.linspace(-0.5, 1.5, 201))
grid = np.c_[g1.ravel(), g2.ravel()]


def shade(ax, yhat_grid, alpha=0.13):
    """Tô vùng đoán 1 (xanh) và vùng đoán 0 (cam) trên lưới g1, g2."""
    ax.contourf(g1, g2, yhat_grid.reshape(g1.shape), levels=[-0.5, 0.5, 1.5], colors=[ORANGE, BLUE], alpha=alpha)


fig, (a, b) = plt.subplots(1, 2, figsize=(9, 4.4))
shade(a, step(step(grid @ W1 + b1) @ W2 + b2)[:, 0])
for k, ls in [(0, "--"), (1, ":")]:
    a.contour(
        g1, g2, (grid @ W1[:, k] + b1[k]).reshape(g1.shape), levels=[0], colors="black", linewidths=1.8, linestyles=ls
    )
a.plot([], [], "k--", lw=1.8, label="đường của h₁ (OR)")
a.plot([], [], "k:", lw=1.8, label="đường của h₂ (AND)")
a.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, fontsize=9)
show_points(a, legend=False)
a.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="x₁", ylabel="x₂", title="Không gian đầu vào: hai đường")

shade(b, step(z_of(grid, w_h, b_h)))
b.contour(g1, g2, z_of(grid, w_h, b_h).reshape(g1.shape), levels=[0], colors=GREEN, linewidths=2.5)
show_points(b, H_hand, Y)
for h in np.unique(H_hand, axis=0):
    count = int((H_hand == h).all(axis=1).sum())
    b.annotate(f"{count} điểm", h, xytext=(12, 10), textcoords="offset points", fontsize=9)
b.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xlabel="h₁ (OR)", ylabel="h₂ (AND)", title="Không gian ẩn: một đường đủ")
for ax in (a, b):
    ax.set_aspect("equal")
    ax.set(xticks=[-0.5, 0, 0.5, 1, 1.5], yticks=[-0.5, 0, 0.5, 1, 1.5])
plt.tight_layout()
vn_axes(plt.gcf())
plt.show()

Đây là ý chính của bài: **lớp ẩn không phân loại; nó đổi cách biểu diễn dữ liệu**, sao cho lớp tiếp theo, vẫn chỉ là
một perceptron, làm được việc của mình. Mạng nhiều lớp (MLP, *multilayer perceptron*) là chồng các bước biến đổi như
vậy.

### 🤔 Dự đoán trước: bỏ hàm kích hoạt
Giữ nguyên $W_1$, $b_1$ nhưng bỏ $f$ ở lớp ẩn: $h = xW_1 + b_1$. Perceptron trên các điểm $h$ mới có giải được XOR không?

In [ ]:
H_lin = P @ W1 + b1  # lớp ẩn không có hàm kích hoạt
lin_run = perceptron(H_lin, Y, [0.3, -0.2], 0.1, max_updates=1000)
print("4 điểm trong không gian ẩn tuyến tính:", "  ".join(f"({vn(h[0], 1)}; {vn(h[1], 1)})" for h in H_lin))
print(f"perceptron trên h: {len(lin_run) - 1} lần sửa, nhiều nhất {max(s[2] for s in lin_run)}/4 điểm đúng")
assert max(s[2] for s in lin_run) < 4

Không. Không có $f$, hai cột của $W_1$ ở đây giống hệt nhau, nên $h_2 = h_1 - 1$: bốn điểm nằm trên **một đường
thẳng** theo thứ tự $(0,0)$, rồi $(0,1)$ và $(1,0)$ trùng nhau, rồi $(1,1)$; nhãn đi 0, 1, 1, 0 và không đường thẳng
nào cắt được như thế. Tổng quát hơn (Bài 02): $(xW_1 + b_1)W_2 + b_2 = x(W_1 W_2) + (b_1 W_2 + b_2)$ vẫn là **một**
phép biến đổi tuyến tính, nên mặt quyết định vẫn là một đường thẳng, dù chọn $W_1$ nào. Chính hàm kích hoạt phi tuyến
mới “bẻ cong” được không gian.

## 6 · Để mạng tự học trọng số

Ở mục 4 ta tự chọn trọng số. Mạng có học ra được chúng không? Quy tắc perceptron không dùng được cho lớp ẩn: lớp ẩn
không có nhãn, nên không biết $h$ “đúng” là gì. Cách hiện đại là:

1. Thay hàm bậc thang bằng **sigmoid** $\sigma(z) = 1/(1 + e^{-z})$ ở cả hai lớp (Bài 02), để đầu ra $\hat y \in (0, 1)$
   thay đổi mượt theo từng trọng số.
2. Đo mức sai bằng một con số, ở đây là sai số bình phương trung bình trên bốn điểm,
   $\mathcal L = \tfrac14 \sum_i (\hat y_i - y_i)^2$ (Bài 04 sẽ bàn hàm mất mát nào tốt hơn).
3. Lặp lại: nhích mỗi tham số một chút theo hướng làm $\mathcal L$ giảm: $\theta \leftarrow \theta - \eta\,
   \partial \mathcal L / \partial \theta$ (gradient descent, Bài 05).

Để không đi trước Bài 06, ta tính đạo hàm $\partial \mathcal L / \partial \theta$ bằng cách **thử**: nhích tham số
$\theta_k$ lên và xuống một lượng $\epsilon = 10^{-4}$ rồi xem $\mathcal L$ đổi bao nhiêu (sai phân trung tâm).
Cách này chậm, nhưng với 9 tham số thì không sao, và ta chạy 100 mạng (100 seed khởi tạo) song song: mảng `theta` có cỡ
100 × 9, mỗi hàng là toàn bộ tham số của một mạng. Mọi tham số khởi tạo từ phân phối chuẩn $N(0, 1)$; ta dùng
$\eta = 4$ và 1.000 bước (Bài 05 bàn cách chọn $\eta$).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def unpack(theta, H):
    """theta (..., 4H + 1) → W₁ (..., 2, H), b₁ (..., H), W₂ (..., H), b₂ (...)."""
    W1 = theta[..., : 2 * H].reshape(*theta.shape[:-1], 2, H)
    return W1, theta[..., 2 * H : 3 * H], theta[..., 3 * H : 4 * H], theta[..., 4 * H]


def mlp(theta, H, X=P):
    """h = σ(xW₁ + b₁), ŷ = σ(hW₂ + b₂) cho mọi hàng của X và mọi mạng trong theta."""
    W1, b1, W2, b2 = unpack(theta, H)
    h = sigmoid(X @ W1 + b1[..., None, :])  # (..., số điểm, H)
    return sigmoid((h @ W2[..., None])[..., 0] + b2[..., None]), h


def loss(theta, H):
    return ((mlp(theta, H)[0] - Y) ** 2).mean(axis=-1)


def num_grad(theta, H, eps=1e-4):
    """∂L/∂θₖ ≈ [L(θ + εeₖ) − L(θ − εeₖ)] / 2ε, cho mọi k cùng lúc."""
    E = eps * np.eye(theta.shape[-1])
    return (loss(theta[..., None, :] + E, H) - loss(theta[..., None, :] - E, H)) / (2 * eps)


def init_theta(H, seeds=range(100), scale=1.0):
    """Một hàng tham số cho mỗi seed: mọi trọng số và độ lệch lấy từ N(0, scale²)."""
    return np.stack([np.random.default_rng(s).normal(0, scale, 4 * H + 1) for s in seeds])


def train_xor(H, seeds=range(100), steps=1000, eta=4.0, scale=1.0, theta=None):
    """Gradient descent cho nhiều mạng 2 → H → 1 cùng lúc. Trả về tham số cuối, đường mất mát, và mạng nào giải được."""
    theta = init_theta(H, seeds, scale) if theta is None else theta
    curve = [loss(theta, H)]
    for _ in range(steps):
        theta = theta - eta * num_grad(theta, H)
        curve.append(loss(theta, H))
    solved = ((mlp(theta, H)[0] >= 0.5) == (Y == 1)).all(axis=-1)  # đúng cả 4 khi lấy ngưỡng ŷ ≥ 0,5
    return theta, np.array(curve), solved


theta2, curve2, solved2 = train_xor(H=2)  # 100 mạng 2 → 2 → 1, mỗi mạng 9 tham số, η = 4, 1.000 bước
yhat0, h0 = mlp(theta2[0], 2)
print(f"mạng seed 0: L = {vn(curve2[0, 0], 4)} lúc đầu → {vn(curve2[-1, 0], 4)} sau {vn(len(curve2) - 1, 0)} bước")
for p, h, yh in zip(P, h0, yhat0):
    print(f"  x = ({p[0]:.0f}, {p[1]:.0f})  →  h = ({vn(h[0])}; {vn(h[1])})  →  ŷ = {vn(yh, 3)}")
assert solved2[0] and curve2[-1, 0] < 0.002

Mạng seed 0 học được XOR: $\hat y$ gần 1 ở hai điểm nhãn 1 và gần 0 ở hai điểm nhãn 0. Nhưng lớp ẩn của nó **không
giống** lời giải OR/AND ta tự đặt: một nơ-ron ẩn bật lên cho $(0,1)$, nơ-ron kia cho $(1,0)$, còn hai điểm nhãn 0 bị
đẩy về gần $(0, 0)$. Cùng một bài toán có nhiều biểu diễn ẩn tốt; gradient descent tìm ra một cái, không nhất thiết là
cái con người nghĩ ra.

Hình dưới vẽ mạng này như hình ở mục 5: bên trái, hai đường nét gạch và nét chấm là nơi mỗi nơ-ron ẩn đổi từ dưới
lên trên 0,5, nền tô theo $\hat y \ge 0{,}5$; bên phải, bốn điểm trong không gian ẩn $(h_1, h_2)$ và đường $\hat y = 0{,}5$
của nơ-ron đầu ra (xanh lá).

In [ ]:
def show_learned(theta, title):
    """Mạng 2 → 2 → 1 đã học: không gian đầu vào (trái) và không gian ẩn (phải), như hình ở mục 5."""
    _, (a, b) = plt.subplots(1, 2, figsize=(9, 4.4))
    yhat_grid, h_grid = mlp(theta, 2, X=grid)
    shade(a, (yhat_grid >= 0.5).astype(int))
    for k, ls in [(0, "--"), (1, ":")]:
        a.contour(g1, g2, h_grid[:, k].reshape(g1.shape), levels=[0.5], colors="black", linewidths=1.8, linestyles=ls)
    show_points(a, legend=False)
    a.set(xlabel="x₁", ylabel="x₂", title=f"{title}: không gian đầu vào")
    _, _, W2_, b2_ = unpack(theta, 2)
    shade(b, step(grid @ W2_ + b2_))  # trong không gian ẩn: ŷ ≥ 0,5 ⇔ hW₂ + b₂ ≥ 0
    b.contour(g1, g2, (grid @ W2_ + b2_).reshape(g1.shape), levels=[0], colors=GREEN, linewidths=2.5)
    show_points(b, mlp(theta, 2)[1], Y)
    b.set(xlabel="h₁", ylabel="h₂", title=f"{title}: không gian ẩn đã học")
    for ax in (a, b):
        ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xticks=[-0.5, 0, 0.5, 1, 1.5], yticks=[-0.5, 0, 0.5, 1, 1.5])
        ax.set_aspect("equal")
    plt.tight_layout()
    vn_axes(plt.gcf())
    plt.show()


show_learned(theta2[0], "Seed 0")

Bên phải, hai điểm cam chụm lại gần gốc, hai điểm xanh nằm ở hai góc kia: một đường thẳng tách được chúng, đúng như
với lời giải tự đặt.

### 🤔 Dự đoán trước
Hai nơ-ron ẩn là **đủ** để biểu diễn XOR (mục 4 vừa làm bằng tay). Vậy trong 100 mạng 2 → 2 → 1 khởi tạo ngẫu nhiên
(100 seed), cùng η và cùng 1.000 bước, bao nhiêu mạng học được XOR?

In [ ]:
lo, hi = curve2[-1].min(), curve2[-1].max()
print(f"giải được XOR: {solved2.sum()}/100 seed;  L cuối từ {vn(lo, 4)} đến {vn(hi, 4)}")
print(f"  các mạng đã giải: L cuối ≤ {vn(curve2[-1, solved2].max(), 4)}")
print(f"  các mạng bị kẹt: L cuối từ {vn(curve2[-1, ~solved2].min(), 4)} đến {vn(curve2[-1, ~solved2].max(), 4)}")
assert abs(solved2.sum() - 77) <= 3, "khác xa kết quả khi soạn notebook (77/100)"

fig, ax = plt.subplots(figsize=(8, 3.4))
steps_axis = np.arange(curve2.shape[0])
for s in range(100):
    ok = solved2[s]
    ax.plot(steps_axis, curve2[:, s], color=BLUE if ok else ORANGE, ls="-" if ok else "--", lw=0.8, alpha=0.6)
ax.plot([], [], color=BLUE, label=f"giải được ({solved2.sum()} seed)")
ax.plot([], [], color=ORANGE, ls="--", label=f"bị kẹt ({(~solved2).sum()} seed)")
ax.set(yscale="log", xlabel="bước gradient descent", ylabel="mất mát L (thang log)", xlim=(0, steps_axis[-1]))
ax.set_title("100 mạng 2 → 2 → 1 trên XOR, mỗi đường một seed")
ax.legend(frameon=False)
vn_axes(plt.gcf())
plt.show()

Gần một phần tư số mạng không học được, dù hai nơ-ron ẩn là đủ. Trên đồ thị, đường xanh (giải được) lao xuống sau vài
trăm bước; đường cam nét gạch (bị kẹt) dừng lại ở một mức gần như nằm ngang và không xuống nữa. **Biểu diễn được không
có nghĩa là học được.**

### Một mạng bị kẹt trông thế nào?
Ô dưới đo, trong không gian ẩn của từng mạng, khoảng cách nhỏ nhất giữa hai điểm **khác nhãn**, và độ lớn gradient
$\lVert \partial \mathcal L / \partial \theta \rVert$ ở cuối so với lúc khởi tạo. Rồi nó cho các mạng bị kẹt chạy thêm
4.000 bước.

In [ ]:
_, h_all = mlp(theta2, 2)  # (100 mạng, 4 điểm, 2 nơ-ron ẩn)
opposite = [(i, j) for i, j in itertools.combinations(range(4), 2) if Y[i] != Y[j]]
gap = np.min([np.linalg.norm(h_all[:, i] - h_all[:, j], axis=-1) for i, j in opposite], axis=0)
g_end = np.linalg.norm(num_grad(theta2, 2), axis=-1)
g_start = np.linalg.norm(num_grad(init_theta(2), 2), axis=-1)
print("khoảng cách nhỏ nhất giữa hai điểm khác nhãn trong không gian ẩn:")
print(f"  mạng giải được: {vn(gap[solved2].min(), 2)} đến {vn(gap[solved2].max(), 2)}")
print(f"  mạng bị kẹt:    {vn(gap[~solved2].min(), 4)} đến {vn(gap[~solved2].max(), 4)}")
print(
    f"|gradient| trung vị: {vn(np.median(g_start), 4)} lúc đầu, {vn(np.median(g_end[~solved2]), 4)} ở các mạng bị kẹt"
)
print(f"  tức nhỏ đi khoảng {vn(np.median(g_start) / np.median(g_end[~solved2]), 0)} lần")
stuck_seeds = np.flatnonzero(~solved2)
near_eighth = curve2[-1, stuck_seeds] < 0.13
other = stuck_seeds[~near_eighth][0]
print(f"{near_eighth.sum()}/{len(stuck_seeds)} mạng bị kẹt có L gần 0,125; mạng còn lại (seed {other}) có ŷ =", end=" ")
print("  ".join(vn(v, 3) for v in mlp(theta2[other], 2)[0]))
theta_more, _, solved_more = train_xor(2, theta=theta2[stuck_seeds], steps=4000)
print(f"chạy thêm 4.000 bước: {solved_more.sum()}/{len(stuck_seeds)} mạng bị kẹt thoát ra;", end=" ")
norm_before, norm_after = np.linalg.norm(theta2[stuck_seeds], axis=1), np.linalg.norm(theta_more, axis=1)
print(f"|θ| trung vị {vn(np.median(norm_before), 1)} → {vn(np.median(norm_after), 1)}")
print(f"seed {stuck_seeds[0]}: ŷ =", "  ".join(vn(v, 3) for v in mlp(theta2[stuck_seeds[0]], 2)[0]))
assert gap[~solved2].max() < 0.05 < gap[solved2].min()

In [ ]:
show_learned(theta2[stuck_seeds[0]], f"Seed {stuck_seeds[0]}")

Các mạng bị kẹt đã **gộp nhầm**. Ở mục 5, lớp ẩn gộp hai điểm *cùng nhãn* $(0,1)$ và $(1,0)$, và đó chính là lời giải.
Ở đây lớp ẩn gộp hai điểm *khác nhãn* vào gần như cùng một chỗ (hình phải: một hình tròn cam và một hình vuông xanh
chồng lên nhau). Nơ-ron đầu ra không thể tách hai điểm trùng nhau, nên trả lời $\hat y \approx 0{,}5$ cho cả hai: hai
sai số $0{,}5^2$ trên bốn điểm cho $\mathcal L \approx 2 \cdot 0{,}25 / 4 = 0{,}125$, đúng mức mà các đường cam dừng lại.
(Mạng kẹt còn lại gộp *ba* điểm, hai cam một xanh, và trả lời $\hat y \approx 1/3$ cho cả ba: $\mathcal L \approx 1/6$.)
Để tách hai điểm đó ra, nơ-ron ẩn phải ra khỏi vùng bão hòa của sigmoid (Bài 02), nơi độ dốc gần 0: gradient nhỏ hơn
lúc đầu hơn trăm lần, nên thêm 4.000 bước chỉ cứu được 1 mạng.

Đây không hẳn là một “cực tiểu địa phương”. Với mạng 2 → 2 → 1 sigmoid trên XOR, Sprinkhuizen-Kuyper & Boers (1998)
chứng minh mọi điểm dừng có trọng số hữu hạn đều là điểm yên ngựa hoặc cực tiểu toàn cục với sai số 0. Mạng bị kẹt
đang trôi rất chậm trên một vùng gần như phẳng, với trọng số lớn dần ($|\theta|$ tăng trong ô trên). Với người huấn
luyện, kết quả thì như nhau: nó không học được trong thời gian có hạn. (Bài 05 bàn về vùng phẳng và điểm yên ngựa.)

## 7 · Nhiều nơ-ron ẩn hơn

### Rộng hơn thì dễ học hơn
Chạy lại đúng thí nghiệm 100 seed với $H$ = 1, 2, 3, 4 nơ-ron ẩn (mỗi mạng có $4H + 1$ tham số):

In [ ]:
width = {}
for H in [1, 2, 3, 4]:
    _, curve_H, solved_H = (theta2, curve2, solved2) if H == 2 else train_xor(H)
    width[H] = int(solved_H.sum())
    print(
        f"H = {H}: {4 * H + 1:>2} tham số, giải được {width[H]:>3}/100 seed, L cuối trung vị {vn(np.median(curve_H[-1]), 4)}"
    )
assert width[1] == 0 and width[4] >= 95

Một nơ-ron ẩn thì không bao giờ đủ: lớp ẩn 1 chiều xếp bốn điểm trên một đường thẳng, và nhãn 0, 1, 1, 0 không cắt được
bằng một ngưỡng. Hai nơ-ron đủ để biểu diễn nhưng hay kẹt; thêm một hai nơ-ron nữa thì hầu như seed nào cũng học được.
Mạng thừa nơ-ron có nhiều cách gộp đúng hơn, nên ít khả năng mọi nơ-ron cùng gộp nhầm. Đây là một lý do mạng thật thường
lớn hơn mức tối thiểu cần để biểu diễn bài toán.

### Nhiều nơ-ron, nhiều đường thẳng, vùng phức tạp hơn
Mỗi nơ-ron ẩn bậc thang là một đường thẳng chia mặt phẳng làm đôi. Nơ-ron đầu ra kết hợp các nửa mặt phẳng đó. Ví dụ:
để nhận ra điểm nằm **trong hình tròn** bán kính 1, đặt $H$ nơ-ron, mỗi nơ-ron là một đường tiếp tuyến của hình tròn,
rồi cho đầu ra bật khi **cả $H$** nơ-ron đều bật (một phép AND, như $h_2$ ở mục 4). Vùng nhận được là một đa giác đều
$H$ cạnh ngoại tiếp hình tròn. Ô dưới đo diện tích chỗ sai (trên một lưới điểm) và so với công thức hình học
$H\tan(\pi/H) - \pi$.

In [ ]:
gx, gy = np.meshgrid(np.linspace(-2.1, 2.1, 841), np.linspace(-2.1, 2.1, 841))  # tam giác có đỉnh ở bán kính 2
plane = np.c_[gx.ravel(), gy.ravel()]
in_disk = (plane**2).sum(axis=1) <= 1
cell_area = (4.2 / 840) ** 2  # diện tích mỗi ô lưới


def polygon_net(H):
    """Lớp ẩn: H đường tiếp tuyến (pháp tuyến hướng vào tâm); đầu ra: AND của cả H nơ-ron."""
    angle = 2 * np.pi * np.arange(H) / H
    W1 = -np.stack([np.cos(angle), np.sin(angle)])  # 2 × H; nơ-ron k bật khi x·uₖ ≤ 1
    b1 = np.ones(H)
    W2, b2 = np.ones(H), -(H - 0.5)  # bật khi cả H nơ-ron ẩn cùng bật
    return lambda X: step(step(X @ W1 + b1) @ W2 + b2)


fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, H in zip(axes, [3, 4, 8, 32]):
    inside = polygon_net(H)(plane)
    err = (inside != in_disk).sum() * cell_area
    exact = H * np.tan(np.pi / H) - np.pi
    print(
        f"H = {H:>2}: diện tích sai {vn(err, 3)} (công thức: {vn(exact, 3)}), tức {pct(exact / np.pi, 1)} diện tích hình tròn"
    )
    ax.contourf(gx, gy, inside.reshape(gx.shape), levels=[0.5, 1.5], colors=[BLUE], alpha=0.35)
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="black", lw=1.2, ls="--"))
    ax.set(xlim=(-2.1, 2.1), ylim=(-2.1, 2.1), xticks=[], yticks=[], title=f"H = {H} nơ-ron ẩn")
    ax.set_aspect("equal")
    assert abs(err - exact) < 0.02
plt.tight_layout()
vn_axes(plt.gcf())
plt.show()

Vùng tô xanh là nơi mạng trả lời “trong”; nét gạch là hình tròn thật. Với 3 nơ-ron, tam giác thừa ra 65,4% diện tích hình
tròn; với 32 nơ-ron, chỉ còn khoảng 0,3%. Càng nhiều nơ-ron, đa giác càng sát hình tròn. Đây là trực giác của **định lý
xấp xỉ phổ quát** (Cybenko 1989; Hornik 1991; Leshno và cộng sự 1993): với hàm kích hoạt không phải đa thức, một lớp ẩn
đủ rộng xấp xỉ được mọi hàm liên tục trên một miền đóng và bị chặn, sai số nhỏ tùy ý.

Định lý chỉ nói một mạng như vậy **tồn tại**. Nó không nói cần bao nhiêu nơ-ron (với hình tròn ở đây, diện tích sai
giảm cỡ $\pi^3 / 3H^2$), và không nói gradient descent có tìm ra trọng số đó hay không: mục 6 vừa cho thấy ngay cả
khi mạng đủ lớn, việc học vẫn có thể kẹt. Ở đây ta tự đặt trọng số, đúng như cách chứng minh định lý.

### Sâu hay rộng?
Thêm nơ-ron vào **một** lớp ẩn là làm mạng rộng hơn. Thêm **lớp** là làm mạng sâu hơn. Với cùng số nơ-ron, cách nào
vẽ được hình phức tạp hơn? Một ví dụ một chiều (Telgarsky, 2015) trả lời rõ ràng. Lấy hàm “mái nhà” trên $[0, 1]$:

$$g(x) = 2\,\mathrm{ReLU}(x) - 4\,\mathrm{ReLU}(x - 0{,}5),$$

tức một lớp ẩn có **2** nơ-ron ReLU: $g$ đi lên từ 0 đến 1 rồi xuống lại 0. Xếp $k$ lớp như vậy chồng lên nhau
($g$ của $g$ của $g$…) thì mỗi lớp gập đôi số đoạn thẳng. Ô dưới đếm số đoạn thẳng của hàm mà mạng tính ra.

In [ ]:
def relu(z):
    return np.maximum(0, z)


def g(x):
    return 2 * relu(x) - 4 * relu(x - 0.5)  # một lớp ẩn, 2 nơ-ron ReLU


x_line = np.linspace(0, 1, 2**12 + 1)  # các điểm k/4096: phép tính trên chúng là chính xác


def n_pieces(y):
    """Số đoạn thẳng của một hàm tuyến tính từng khúc, đếm qua các chỗ độ dốc đổi giá trị."""
    slope = np.diff(y) / np.diff(x_line)
    return 1 + int((np.diff(slope) != 0).sum())


fig, axes = plt.subplots(1, 3, figsize=(10, 2.6), sharey=True)
y_line = x_line
for k in range(1, 7):
    y_line = g(y_line)  # thêm một lớp
    print(
        f"{k} lớp × 2 nơ-ron = {2 * k:>2} nơ-ron: {n_pieces(y_line):>2} đoạn thẳng;  1 lớp cần ≥ {2**k - 1:>2} nơ-ron"
    )
    assert n_pieces(y_line) == 2**k
    if k in (1, 2, 4):
        ax = axes[[1, 2, 4].index(k)]
        ax.plot(x_line, y_line, color=BLUE, lw=1.6)
        ax.set(xlabel="x", title=f"{k} lớp ({2 * k} nơ-ron): {2**k} đoạn")
axes[0].set_ylabel("đầu ra")
plt.tight_layout()
vn_axes(plt.gcf())
plt.show()

Với $k$ lớp, mỗi lớp 2 nơ-ron, hàm có $2^k$ đoạn thẳng. Còn một lớp ẩn duy nhất với $m$ nơ-ron ReLU thì cho tối đa
$m + 1$ đoạn trên một đường thẳng, vì mỗi nơ-ron chỉ tạo thêm một chỗ gập. Muốn có 64 đoạn như 6 lớp × 2 nơ-ron
(12 nơ-ron), mạng một lớp cần ít nhất 63 nơ-ron. **Độ sâu nhân số vùng, độ rộng chỉ cộng.**
Telgarsky chứng minh có những hàm mà mạng sâu tính chính xác, còn mọi mạng nông cần số nơ-ron tăng theo hàm mũ mới
xấp xỉ nổi. Điều đó không có nghĩa là sâu luôn tốt hơn (mạng sâu khó huấn luyện hơn: Chương 3 của khóa học), nhưng giải
thích vì sao mạng thực tế có nhiều lớp thay vì một lớp thật rộng.

## 8 · MLP thật: mạng MNIST 784 → 128 → 64 → 10

Mạng đi cùng ta qua các bài tiếp theo (ví dụ MNIST của MLSysBook) là một MLP có hai lớp ẩn, cùng quy ước vectơ hàng:

$$h_1 = \mathrm{ReLU}(x W_1 + b_1), \qquad h_2 = \mathrm{ReLU}(h_1 W_2 + b_2), \qquad z = h_2 W_3 + b_3$$

$x$ là ảnh 784 điểm ảnh; $z$ gồm 10 điểm số, một cho mỗi chữ số, và mạng chọn chữ số có điểm cao nhất (Bài 04 biến
10 điểm số đó thành xác suất bằng softmax). Đếm phép MAC và tham số:

In [ ]:
SIZES = [784, 128, 64, 10]
for n_in, n_out in itertools.pairwise(SIZES):
    print(
        f"lớp {n_in:>3} → {n_out:>3}: W cỡ {n_in}×{n_out}, {vn(n_in * n_out, 0):>7} MAC, {vn(n_in * n_out + n_out, 0):>7} tham số"
    )
macs = sum(n_in * n_out for n_in, n_out in itertools.pairwise(SIZES))
n_par = sum(n_in * n_out + n_out for n_in, n_out in itertools.pairwise(SIZES))
print(f"tổng: {vn(macs, 0)} MAC cho mỗi ảnh, {vn(n_par, 0)} tham số")
print(f"num_grad của mục 6 sẽ cần {vn(2 * n_par, 0)} lần chạy mạng cho mỗi bước học")
assert macs == 109184 and n_par == 109386

109.184 MAC, khớp với trang bài học. Ô dưới huấn luyện mạng đúng như `reel/mnist_mlp/train.py` của khóa học (cùng seed,
nên cùng con số với demo các bài sau): khởi tạo He (Bài 09), mỗi bước một lô 64 ảnh, $\eta = 0{,}1$, 5 lượt. Có hai điều
khác mục 6:

- Đầu ra dùng softmax và cross-entropy thay cho sigmoid và sai số bình phương (Bài 04 giải thích vì sao).
- Gradient không tính bằng cách thử: như ô trên vừa in, `num_grad` cần hàng trăm nghìn lần chạy mạng cho *mỗi* bước.
  Ba dòng đánh dấu “lan truyền ngược” tính đúng gradient đó với chi phí cỡ một lần chạy xuôi. Bài 06 giải thích chúng;
  ở đây cứ dùng như một hộp đen.

In [ ]:
def forward(params, X):
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = relu(X @ W1 + b1)
    h2 = relu(h1 @ W2 + b2)
    return h1, h2, h2 @ W3 + b3


def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


rng = np.random.default_rng(0)
params = [
    [rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)), np.zeros(n_out)] for n_in, n_out in itertools.pairwise(SIZES)
]
params0 = [[W.copy(), b.copy()] for W, b in params]  # giữ lại mạng lúc chưa học
for epoch in range(1, 6):
    order = rng.permutation(len(Xtr))
    for i in range(0, len(order), 64):
        idx = order[i : i + 64]
        x, y = Xtr[idx], ytr[idx]
        h1, h2, z = forward(params, x)
        dz = softmax(z)
        dz[np.arange(len(y)), y] -= 1
        dz /= len(y)
        (W1_, _), (W2_, _), (W3_, _) = params
        dh2 = (dz @ W3_.T) * (h2 > 0)  # lan truyền ngược (Bài 06)
        dh1 = (dh2 @ W2_.T) * (h1 > 0)  # lan truyền ngược
        grads = [(x.T @ dh1, dh1.sum(0)), (h1.T @ dh2, dh2.sum(0)), (h2.T @ dz, dz.sum(0))]  # lan truyền ngược
        for layer, (gW, gb) in zip(params, grads):
            layer[0] -= 0.1 * gW
            layer[1] -= 0.1 * gb
    acc = (forward(params, Xte)[2].argmax(axis=1) == yte).mean()
    print(f"lượt {epoch}: đúng {pct(acc)} trên {vn(len(yte), 0)} ảnh kiểm tra")
assert abs(acc - 0.9727) < 0.0005, "khác với demo các bài sau (97,27%): hãy kiểm tra lại dữ liệu"

97,27% sau 5 lượt, đúng con số của demo các bài sau; perceptron 10 lớp của Bài 01 dừng ở 88,70% (ô kế tiếp chạy lại).

### Lớp ẩn của MNIST đã học được gì?
Ở XOR ta nhìn được không gian ẩn bằng mắt. Ở MNIST, $h_1$ có 128 chiều, nên ta đo nó: cho **perceptron 10 lớp của
Bài 01** (một lượt, $\eta = 0{,}1$, seed 0) học trên các đặc trưng khác nhau thay cho pixel. Nếu một lớp ẩn làm dữ liệu
“tách được hơn”, perceptron đơn giản đặt trên nó sẽ đúng nhiều hơn.

### 🤔 Dự đoán trước
Trên pixel, perceptron đúng 88,70% (Bài 01). Trên $h_1$ của **mạng chưa học** (128 nơ-ron ReLU, trọng số ngẫu nhiên lúc
khởi tạo), nó đúng nhiều hơn hay ít hơn? Còn trên $h_1$ và $h_2$ của mạng đã học?

In [ ]:
def perceptron10(F, y, F_test, y_test, seed=0, eta=0.1):
    """Perceptron 10 lớp của Bài 01: sai thì cộng ηx vào cột chữ số đúng và trừ ηx khỏi cột chữ số bị đoán nhầm."""
    W, b, fixes = np.zeros((F.shape[1], 10)), np.zeros(10), 0
    for i in np.random.default_rng(seed).permutation(len(y)):
        k = int(np.argmax(F[i] @ W + b))
        if k != y[i]:
            W[:, y[i]] += eta * F[i]
            b[y[i]] += eta
            W[:, k] -= eta * F[i]
            b[k] -= eta
            fixes += 1
    return fixes, ((F_test @ W + b).argmax(axis=1) == y_test).mean()


h1_tr, h2_tr, _ = forward(params, Xtr)
h1_te, h2_te, _ = forward(params, Xte)
features = {
    "pixel x (784 chiều)": (Xtr, Xte),
    "h₁ của mạng chưa học (128)": (forward(params0, Xtr)[0], forward(params0, Xte)[0]),
    "h₁ của mạng đã học (128)": (h1_tr, h1_te),
    "h₂ của mạng đã học (64)": (h2_tr, h2_te),
}
probe = {}
for name, (F, F_test) in features.items():
    fixes, probe[name] = perceptron10(F, ytr, F_test, yte)
    print(f"perceptron trên {name:<28}: {vn(fixes, 0):>6} lần sửa, đúng {pct(probe[name])}")
assert abs(probe["pixel x (784 chiều)"] - 0.887) < 0.002, "khác với Bài 01 (88,7%)"
assert probe["h₁ của mạng chưa học (128)"] < probe["pixel x (784 chiều)"] < probe["h₂ của mạng đã học (64)"]

Lớp ẩn ngẫu nhiên còn **kém hơn** pixel (85,78% so với 88,70%): tự nó không làm dữ liệu dễ tách hơn. Sau khi học,
perceptron đặt trên $h_1$ hay $h_2$ đúng 96,16–96,45%, gần với 97,27% của cả mạng, dù nó chỉ học bằng quy tắc của Bài 01
trong một lượt. Phần việc khó đã nằm ở các lớp ẩn: chúng biến 10 lớp chữ số thành những đám điểm gần như tách được bằng
các siêu phẳng, đúng như lớp ẩn của XOR biến bốn điểm thành ba điểm tách được.

Cặp chữ số khó của Bài 01, 4 và 9, cho thấy điều đó rõ hơn. Ô dưới chạy lại đúng perceptron hai lớp của Bài 01 trên
pixel và trên $h_2$, rồi chiếu ảnh kiểm tra lên **hai hướng mà dữ liệu trải rộng nhất** (phân tích thành phần chính,
PCA; Bài 19 dùng lại) để nhìn được bằng mắt.

In [ ]:
def perceptron_pass(X, y, eta=0.1, seed=0):
    """Perceptron hai lớp của Bài 01: w = 0, một lượt theo thứ tự ngẫu nhiên, chỉ sửa khi sai."""
    w, b, fixes = np.zeros(X.shape[1]), 0.0, 0
    for i in np.random.default_rng(seed).permutation(len(y)):
        y_hat = int(X[i] @ w + b >= 0)
        if y_hat != y[i]:
            w, b, fixes = w + eta * (y[i] - y_hat) * X[i], b + eta * (y[i] - y_hat), fixes + 1
    return w, b, fixes


keep_tr, keep_te = np.isin(ytr, [4, 9]), np.isin(yte, [4, 9])
y49, y49_te = (ytr[keep_tr] == 4).astype(int), (yte[keep_te] == 4).astype(int)  # y = 1 nghĩa là "4"
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (name, F, F_te) in zip(
    axes, [("pixel x", Xtr[keep_tr], Xte[keep_te]), ("h₂ của mạng đã học", h2_tr[keep_tr], h2_te[keep_te])]
):
    w, b, fixes = perceptron_pass(F, y49)
    n_ok = int(((F_te @ w + b >= 0) == y49_te).sum())
    print(
        f"4 hay 9 trên {name}: {fixes} lần sửa, kiểm tra đúng {vn(n_ok, 0)}/{vn(len(y49_te), 0)} = {pct(n_ok / len(y49_te))}"
    )
    if name == "pixel x":
        assert (fixes, n_ok) == (798, 1889), "khác với Bài 01"
    C = F_te - F_te.mean(axis=0)
    proj = C @ np.linalg.svd(C, full_matrices=False)[2][:2].T  # tọa độ trên hai thành phần chính
    for lab, color, marker, txt in [(1, ORANGE, "o", "số 4"), (0, BLUE, "s", "số 9")]:
        sel = y49_te == lab
        ax.scatter(*proj[sel].T, s=7, color=color, marker=marker, alpha=0.45, lw=0, label=txt)
    ax.set(xlabel="thành phần chính 1", ylabel="thành phần chính 2", xticks=[], yticks=[], title=name)
    ax.legend(frameon=False, markerscale=2.5, fontsize=9)
plt.tight_layout()
vn_axes(plt.gcf())
plt.show()

Mỗi chấm là một ảnh kiểm tra (tròn cam: số 4, vuông xanh: số 9). Trên pixel (trái), hai đám chồng lên nhau; trong
không gian ẩn $h_2$ (phải), chúng gần như nằm hai phía. Cùng một perceptron, số lần sửa giảm từ 798 xuống chưa tới
một phần ba (218) và độ chính xác tăng từ 94,88% lên 98,49%. Hai chiều chỉ là một lát cắt của 64 chiều, nhưng ý nghĩa thì
giống XOR: lớp ẩn đã học một **biểu diễn** mà trên đó bài toán gần như tuyến tính.

## 9 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Độ lớn khởi tạo
Mục 6 khởi tạo mọi tham số từ $N(0, 1)$. Đổi độ lệch chuẩn đó (`scale`) cho 100 mạng 2 → 2 → 1:

In [ ]:
for scale in [0.1, 0.5, 1.0, 2.0, 4.0]:
    _, curve_s, solved_s = train_xor(2, scale=scale)
    print(
        f"khởi tạo N(0; {vn(scale, 1)}²): giải được {solved_s.sum():>2}/100 seed, L cuối trung vị {vn(np.median(curve_s[-1]), 4)}"
    )

Khởi tạo quá nhỏ: mọi $z$ gần 0, nên mọi nơ-ron ẩn cho gần 0,5 với cả bốn điểm, gradient rất nhỏ, và 1.000 bước chưa đủ
để bắt đầu (L trung vị 0,2500, đúng mức của $\hat y = 0{,}5$ cho mọi điểm). Khởi tạo quá lớn: sigmoid bão hòa ngay từ đầu,
nhiều mạng gộp nhầm điểm trước khi kịp học. Chọn độ lớn khởi tạo là cả một chủ đề (khởi tạo He, Bài 09).

### Lớp ẩn ngẫu nhiên, như máy Mark I
Perceptron Mark I của Rosenblatt (1958–1960) thật ra **có** một lớp ẩn: 400 tế bào quang điện nối ngẫu nhiên, qua một
bảng cắm dây, tới 512 “đơn vị liên kết”. Các dây nối đó cố định; chỉ lớp cuối được học. Ô dưới làm lại ý đó trên
MNIST: một lớp ẩn 512 nơ-ron ReLU với trọng số ngẫu nhiên không bao giờ học, rồi perceptron 10 lớp đặt lên trên, 3 seed.

In [ ]:
mark1 = []
for seed in range(3):
    R = np.random.default_rng(seed).normal(0, np.sqrt(2 / 784), (784, 512))  # lớp ẩn ngẫu nhiên, cố định
    fixes, acc_r = perceptron10(relu(Xtr @ R), ytr, relu(Xte @ R), yte)
    mark1.append(acc_r)
    print(f"seed {seed}: lớp ẩn ngẫu nhiên 512 nơ-ron → {vn(fixes, 0)} lần sửa, đúng {pct(acc_r)}")
print(f"khoảng giữa các seed: {pct(min(mark1))} – {pct(max(mark1))}  (pixel: {pct(probe['pixel x (784 chiều)'])})")

Với 512 nơ-ron, lớp ẩn ngẫu nhiên giúp một chút so với pixel (89,27–90,98% qua 3 seed, so với 88,70%), nhưng còn xa
lớp ẩn 128 nơ-ron **đã học** (96,45%). Đây đúng là giới hạn mà Minsky & Papert phân tích: một lớp học được đặt trên
các đặc trưng cố định. Thử tăng số nơ-ron ngẫu nhiên (`512`) để xem cần bao nhiêu mới đuổi kịp.

## 10 · Giới hạn và cầu nối sang Bài 04

Bài này đã dùng hai thứ “tạm”:

- **Đầu ra và mất mát.** Với XOR, một sigmoid và sai số bình phương là đủ. Mạng MNIST có 10 đầu ra; để đọc 10 điểm số
  đó như xác suất và đo mạng sai *bao nhiêu* cho từng ảnh, ta cần softmax và cross-entropy, những thứ ô huấn luyện ở
  mục 8 dùng mà chưa giải thích. Đó là Bài 04.
- **Gradient.** Cách “thử nhích từng tham số” của mục 6 cần 218.772 lần chạy mạng MNIST cho mỗi bước. Bài 05 (đi xuống
  dốc) và Bài 06 (lan truyền ngược) thay nó bằng cách tính chính xác và rẻ.

## Tóm tắt
- Không đường thẳng nào tách được XOR (bốn bất đẳng thức mâu thuẫn); quy tắc perceptron rơi vào một vòng lặp 4 bước
  từ lần sửa thứ 11 và không bao giờ quá 3/4.
- Lớp ẩn $h = f(xW_1 + b_1)$ là hai (hay nhiều) perceptron cạnh nhau. Lời giải OR/AND gộp $(0,1)$ và $(1,0)$ thành một
  điểm, và trong không gian ẩn perceptron của Bài 01 giải XOR sau 2 lần sửa.
- Không có $f$ phi tuyến, lớp ẩn vô dụng: hai lớp tuyến tính vẫn là một đường thẳng.
- Mạng 2 → 2 → 1 đủ để biểu diễn XOR nhưng chỉ 77/100 seed học được; số còn lại gộp nhầm hai điểm khác nhãn và kẹt ở
  $\mathcal L \approx 0{,}125$. Với 4 nơ-ron ẩn, 100/100.
- Nhiều nơ-ron vẽ vùng phức tạp hơn (đa giác 32 cạnh sát hình tròn: xấp xỉ phổ quát); nhiều lớp nhân số vùng
  ($2^k$ đoạn với $2k$ nơ-ron, so với cần $2^k - 1$ nơ-ron trong một lớp).
- MLP MNIST 784 → 128 → 64 → 10: 109.184 MAC, 97,27% sau 5 lượt. Perceptron của Bài 01 đặt trên lớp ẩn đã học đúng
  96,45%, trên pixel 88,70%, trên lớp ẩn ngẫu nhiên 85,78%.

## Câu hỏi ôn tập
1. Viết bốn bất đẳng thức mà một perceptron phải thỏa để giải XOR, và chỉ ra vì sao chúng mâu thuẫn.
2. Trong lời giải tự đặt ở mục 4, lớp ẩn làm gì với hai điểm $(0,1)$ và $(1,0)$? Vì sao điều đó đủ để một đường thẳng
   tách được dữ liệu?
3. Bỏ hàm kích hoạt ở lớp ẩn thì mạng 2 → 2 → 1 còn giải được XOR không, dù chọn $W_1$ nào? Vì sao?
4. Hai nơ-ron ẩn đủ để biểu diễn XOR. Vì sao 23 trong 100 mạng vẫn không học được, và vì sao thêm nơ-ron ẩn giúp?
5. Một lớp ẩn $m$ nơ-ron ReLU tạo được nhiều nhất bao nhiêu đoạn thẳng trên một đường thẳng? Còn 6 lớp, mỗi lớp
   2 nơ-ron?
6. Định lý xấp xỉ phổ quát bảo đảm điều gì, và **không** bảo đảm điều gì?

<details><summary>Gợi ý đáp án</summary>

1. $b < 0$; $w_2 + b \ge 0$; $w_1 + b \ge 0$; $w_1 + w_2 + b < 0$. Cộng hai bất đẳng thức giữa được
   $w_1 + w_2 + b \ge -b > 0$, trái với bất đẳng thức cuối.
2. Gộp cả hai vào cùng điểm $h = (1, 0)$. Bốn điểm chỉ còn ba: $(0,0)$ và $(1,1)$ nhãn 0, $(1,0)$ nhãn 1, và một
   đường thẳng tách được điểm giữa khỏi hai điểm kia.
3. Không. $(xW_1 + b_1)W_2 + b_2 = x(W_1W_2) + (b_1W_2 + b_2)$ vẫn là một phép biến đổi tuyến tính của $x$, nên mặt
   quyết định vẫn là một đường thẳng.
4. Chúng gộp nhầm hai điểm khác nhãn vào gần như một chỗ; sigmoid ở đó bão hòa nên gradient nhỏ hơn hơn trăm lần và
   mạng gần như đứng yên ở $\mathcal L \approx 0{,}125$. Với nhiều nơ-ron hơn, có nhiều cách gộp đúng, và ít khả năng
   mọi nơ-ron cùng gộp nhầm.
5. $m + 1$ đoạn (mỗi nơ-ron thêm một chỗ gập). 6 lớp × 2 nơ-ron: $2^6 = 64$ đoạn; một lớp cần ít nhất 63 nơ-ron.
6. Một lớp ẩn đủ rộng, với hàm kích hoạt không phải đa thức, xấp xỉ được mọi hàm liên tục trên một miền đóng và bị chặn.
   Định lý không nói cần bao nhiêu nơ-ron, và không nói gradient descent sẽ tìm ra các trọng số đó.
</details>

## Tìm hiểu thêm
- M. Minsky & S. Papert (1969), [Perceptrons](https://doi.org/10.7551/mitpress/11301.001.0001);
  tóm tắt và tranh luận quanh cuốn sách: [Wikipedia](https://en.wikipedia.org/wiki/Perceptrons_(book)).
- W. McCulloch & W. Pitts (1943), [A logical calculus of the ideas immanent in nervous activity](https://doi.org/10.1007/BF02478259):
  mạng nơ-ron ngưỡng tính được mọi biểu thức logic.
- H. D. Block & S. A. Levin (1970), [On the boundedness of an iterative procedure for solving a system of linear
  inequalities](https://doi.org/10.1090/S0002-9939-1970-0265383-5): định lý xoay vòng của perceptron.
- D. Rumelhart, G. Hinton & R. Williams (1986), [Learning representations by back-propagating
  errors](https://doi.org/10.1038/323533a0).
- G. Cybenko (1989), [Approximation by superpositions of a sigmoidal function](https://doi.org/10.1007/BF02551274);
  K. Hornik (1991), [Approximation capabilities of multilayer feedforward networks](https://doi.org/10.1016/0893-6080(91)90009-T);
  M. Leshno và cộng sự (1993), [hàm kích hoạt không phải đa thức](https://doi.org/10.1016/S0893-6080(05)80131-5).
- I. Sprinkhuizen-Kuyper & E. Boers (1998), [The error surface of the 2-2-1 XOR network](https://doi.org/10.1016/S0893-6080(98)00014-8):
  không có cực tiểu địa phương với trọng số hữu hạn.
- M. Telgarsky (2015), [Representation benefits of deep feedforward networks](https://arxiv.org/abs/1509.08101): độ sâu
  nhân số vùng.
- Goodfellow, Bengio & Courville, [Deep Learning, mục 6.1](https://www.deeplearningbook.org/contents/mlp.html): XOR với
  một lớp ẩn ReLU. M. Nielsen, [Chương 4](http://neuralnetworksanddeeplearning.com/chap4.html): chứng minh trực quan
  của định lý xấp xỉ phổ quát.
- [TensorFlow Playground](https://playground.tensorflow.org/): đổi số nơ-ron và số lớp, xem vùng quyết định thay đổi.
- Máy Mark I: [Wikipedia, Perceptron](https://en.wikipedia.org/wiki/Perceptron). Ví dụ MNIST: MLSysBook,
  [*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html).

**Bài tiếp theo:** [Bài 04 · Softmax & hàm mất mát](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-mat-mat/): biến
10 điểm số thành xác suất, và đo mạng đang sai bao nhiêu.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")